**Q1 — Required Libraries Install**




In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes "torchao>=0.16.0"

**Q2 — GPU Check**

In [2]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No GPU available")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB


**Q3 — Select a Model**

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Model loaded successfully!")
print("Model:", model_name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct


**Q4 — Create a Dataset**

In [4]:
from datasets import Dataset

data = [
    {"question": "What is a computer?", "answer": "A computer is an electronic device that processes data and performs tasks."},
    {"question": "What is a CPU?", "answer": "A CPU is the main processor of a computer that executes instructions."},
    {"question": "What is RAM?", "answer": "RAM is temporary memory used by a computer to store data while programs are running."},
    {"question": "What is an SSD?", "answer": "An SSD is a fast storage device used to store files, programs, and the operating system."},
    {"question": "What is an HDD?", "answer": "An HDD is a storage device that uses magnetic disks to store data."},
    {"question": "What is a motherboard?", "answer": "A motherboard is the main circuit board that connects the computer components."},
    {"question": "What is a GPU?", "answer": "A GPU processes graphics and helps a computer handle visual tasks."},
    {"question": "Why is RAM important?", "answer": "RAM helps the computer run programs smoothly and handle multiple tasks."},
    {"question": "Which is faster, SSD or HDD?", "answer": "An SSD is generally faster than an HDD."},
    {"question": "What is a monitor?", "answer": "A monitor is a display device that shows images and information from a computer."},
    {"question": "What is a keyboard?", "answer": "A keyboard is an input device used to enter text and commands."},
    {"question": "What is a mouse?", "answer": "A mouse is an input device used to control the pointer on a computer screen."},
    {"question": "What is a laptop?", "answer": "A laptop is a portable computer with a built-in screen, keyboard, and battery."},
    {"question": "What is a desktop computer?", "answer": "A desktop computer is a computer designed to be used at a fixed location."},
    {"question": "What is a power supply?", "answer": "A power supply provides electrical power to the computer components."},
    {"question": "What is a computer case?", "answer": "A computer case houses and protects the internal computer components."},
    {"question": "What is Windows?", "answer": "Windows is an operating system developed by Microsoft."},
    {"question": "What is an operating system?", "answer": "An operating system manages computer hardware and software resources."},
    {"question": "What is a computer virus?", "answer": "A computer virus is malicious software that can damage or disrupt a computer."},
    {"question": "How can I protect my computer?", "answer": "Use security software, keep the system updated, and avoid suspicious files and links."},
    {"question": "What is a gaming computer?", "answer": "A gaming computer is designed with hardware suitable for playing demanding games."},
    {"question": "What is a webcam?", "answer": "A webcam is a camera used for video calls and recording."},
    {"question": "What is a printer?", "answer": "A printer produces physical copies of digital documents and images."},
    {"question": "What is Wi-Fi?", "answer": "Wi-Fi is a wireless technology used to connect devices to a network."},
    {"question": "What is Bluetooth?", "answer": "Bluetooth is a wireless technology used to connect nearby devices."},
    {"question": "What is a USB port?", "answer": "A USB port is used to connect compatible devices and transfer data or power."},
    {"question": "What is computer storage?", "answer": "Computer storage is used to permanently save files, programs, and other data."},
    {"question": "What should I check before buying a laptop?", "answer": "Check the processor, RAM, storage, display, battery life, and price."},
    {"question": "Why should I buy from a computer shop?", "answer": "A computer shop can provide products, technical advice, warranty, and support."},
    {"question": "What is a graphics card?", "answer": "A graphics card is a hardware component that processes and displays graphics."}
]

dataset = Dataset.from_list(data)

print("Dataset created successfully!")
print("Number of Q&A pairs:", len(dataset))

Dataset created successfully!
Number of Q&A pairs: 30


**Q5 — Configure LoRA**

In [5]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

print("LoRA configuration created successfully!")

LoRA configuration created successfully!


**Q6 — TRAIN THE MODEL**

In [7]:
!pip install -q trl

In [9]:
formatted_dataset = dataset.map(
    lambda x: {
        "text": f"Question: {x['question']}\nAnswer: {x['answer']}"
    }
)

print("Dataset formatted successfully!")

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Dataset formatted successfully!


In [11]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=True,
    bf16=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=lora_config,
    args=training_args
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Step,Training Loss
1,2.805026
2,3.020627
3,2.410734
4,2.514107


TrainOutput(global_step=4, training_loss=2.6876232624053955, metrics={'train_runtime': 8.8483, 'train_samples_per_second': 3.39, 'train_steps_per_second': 0.452, 'total_flos': 1664589712896.0, 'train_loss': 2.6876232624053955, 'epoch': 1.0})

**Q7 — TEST THE MODEL**

In [14]:
import torch
from transformers import AutoModelForCausalLM

# Load original model
original_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

test_questions = [
    "What is a CPU?",
    "Which is faster, SSD or HDD?",
    "What should I check before buying a laptop?",
    "What is the difference between RAM and storage?"
]

def generate_answer(model, question):
    messages = [
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Get input IDs
    if hasattr(inputs, "input_ids"):
        input_ids = inputs.input_ids.to(model.device)
    else:
        input_ids = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=80,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return answer.strip()


for question in test_questions:
    print("\nQuestion:", question)

    print("Original Model:")
    print(generate_answer(original_model, question))

    print("\nFine-tuned Model:")
    print(generate_answer(trainer.model, question))

    print("-" * 60)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Question: What is a CPU?
Original Model:


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


A CPU (Central Processing Unit) is the brain of a computer or other electronic device that performs calculations and logical operations to process data. It is responsible for executing instructions stored in memory and performing arithmetic, logic, and control functions.

The CPU consists of several components:

1. **Core**: The smallest unit of functionality within the CPU.
2. **L3 Cache**: A small amount of temporary storage

Fine-tuned Model:
A
------------------------------------------------------------

Question: Which is faster, SSD or HDD?
Original Model:
When it comes to speed and performance, both solid-state drives (SSDs) and hard disks (HDDs) have their own advantages and disadvantages.

**Hard Disk Drives (HDD):**
- **Advantages:**
  - **Durability:** HDDs can withstand more wear and tear than SSDs.
  - **Cost:** Generally cheaper compared to SSDs.
  - **

Fine-tuned Model:
When the code/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API/API